# Computer Vision 2026-2027 - NOVA FCT

## Project Stage 1 - Shot boundary detection and recurring scenes

**Authors**:
- Student number, name, email
- Student number, name, email

## 1. Introduction



## 2. Part 1 - Frame dissimilarity and hard cuts

In [18]:
import sys
sys.path.insert(0, "experiments")

import tp1, metrics, scene_grouping as sg
import part3_exp
from render import show


## 3. Part 2 - Gradual transitions and false alarms


## 4. Part 3 - Recurring scenes

### 4.1 Task 5: Shot representation by keyframes

Each shot has tens to hundreds of frames. Comparing every frame of one shot
with every frame of another is inefficient, so we reduce each shot to a small 
set of **keyframes**:

| Strategy | Rule | # keyframes |
|---|---|---|
| `middle` | `frames[(start + end) // 2]` | 1 |
| `uniform` | `K` frames evenly spaced via `np.linspace(start, end, K)` | K (3–5) |

`np.unique` on the indices avoids duplicates when a shot is shorter than `K`
(e.g. a 3-frame shot with K=10 returns only 3 keyframes).

**Choice of K.** Two constraints bound the initial range, chosen as a
*prior* before any results are seen:

- **Coverage** A shot usually has three visually distinct
  phases.
  K=3 is the smallest value that samples all three
  and K<3 has no redundancy against an outlier keyframe. This is a
  hypothesis about how shots are structured; the K-sweep of
  Section 5.1 tests it directly.
- **Cost.** The number of keyframe pairs between two shots grows as
  K², making SIFT much slower as K grows and risking the 5-minute
  budget.

We started with **K=3** and tune it in the K-sweep of Section 5.1
(K ∈ {1, 3, 5, 8}). As Section 5.1.1 shows, the "three phases"
heuristic is **not** confirmed by the data. K=1 already ties K=3 on
mean F1 and the sweep selects K=5 for a different reason: its plateau 
is the smallest K whose plateau passes the robustness criterion introduced 
in Section 5.1.4.

**Aggregating keyframe similarities.** For two shots with `K_a` and `K_b`
keyframes we get `K_a × K_b` pairwise similarities. These are collapsed into
one shot-to-shot value with:

- **`max`** — the strongest pair and the *Default for SIFT*
- **`mean`** — average
- **`vote`** — count of pairs with similarity > 0.5

The final choice is validated empirically in two places: the aggregator
sweep of Section 5.1.2 (which isolates the aggregator with everything
else fixed) and the 2×2 comparison of Section 5.2 (which tests the
aggregator inside the full pipeline).

![Middle vs Uniform keyframes](./images/t5_compare_shot0_dev_01.png)

*Figure 1 — Shot 0 of `dev_01` represented by `middle` (left, 1 frame) and `uniform K=3` (right, 3 frames). The uniform strategy samples the shot across its visual phases; the middle strategy captures only the central state. The K-sweep in 5.1 measures whether the extra coverage translates into better discrimination.*

**Implementation validation.** Two families of tests were run before
integrating the function into the pipeline:

- `aggregate_similarities` — 9 input→output cases covering the three
  aggregators across three list shapes (3-element, empty, 1-element),
  plus an unknown-aggregator case that must raise `ValueError`. All
  passed.
- `extract_keyframes` — two edge cases: (i) a 3-frame shot with `K=10`
  returns only 3 keyframes, confirming `np.unique` caps the output;
  (ii) an unknown strategy raises `ValueError` rather than silently
  returning `None`. Both passed.

### 4.2 Task 6: Shot similarity

#### 4.2.1 Colour similarity

Each keyframe is described by a 3D histogram. Similarity is one minus the
χ² distance:

$$
\chi^2(h_1, h_2) = \tfrac{1}{2}\sum_i \frac{(h_1[i] - h_2[i])^2}{h_1[i] + h_2[i]},
\qquad \mathrm{sim} = 1 - \chi^2 \in [0, 1]
$$

Both histograms are L1-normalised, so χ² ∈ [0, 1] and sim = 1
iff the distributions are identical. A small ε = 1e-10 in the denominator
avoids 0/0 on empty bins.

**Bin resolution** Each frame has 320 × 180 = 57 600
pixels, and a 3D histogram with `B` bins has on average 57 600 / B pixels
per bin. We sweep over four resolutions on both
development videos to choose the optimal number of bins:

In [19]:
show(part3_exp.table_bins_single())

| Video | bins | same | diff | gap | pixels/bin |
|---|---|---|---|---|---|
| dev_01 | 8×8×8 | 0.5900 | 0.3777 | 0.2123 | 112.5 |
| dev_01 | 16×16×16 | 0.5079 | 0.3023 | 0.2056 | 14.1 |
| dev_01 | 30×32×32 | 0.4413 | 0.2598 | 0.1815 | 1.9 |
| dev_01 | 60×64×64 | 0.3639 | 0.2324 | 0.1316 | 0.23 |
| dev_02 | 8×8×8 | 0.8724 | 0.5862 | 0.2862 | 112.5 |
| dev_02 | 16×16×16 | 0.7874 | 0.4943 | 0.2931 | 14.1 |
| dev_02 | 30×32×32 | 0.6973 | 0.4221 | 0.2751 | 1.9 |
| dev_02 | 60×64×64 | 0.5677 | 0.3296 | 0.2381 | 0.23 |


The gap shrinks as bins become finer beyond 16³ on both videos
(on dev_02 it peaks at 16³ (0.2931) before declining to 0.2751 at
`30×32²` and 0.2381 at `60³`). `8³` and `16³` are **practically
tied** on the mean gap (0.2493 vs 0.2494, a difference of 0.0001 —
well below any meaningful threshold) and we adopt `8³` because it
is cheaper to compute than `16³`.

`16³` is kept as a candidate to re-evaluate in the bin-resolution sweep
(Section 5.1.5), where every pair of shots is evaluated instead of a
single pair.

**Colour space: HSV vs. RGB.** The theoretical argument for HSV is that a
flash brightens the three RGB channels together (shifting every bin)
while in HSV only V shifts and H and S stay, so the metric is robust to
illumination changes. On the single test pair, the gap supports this
expectation:


In [20]:
show(part3_exp.table_colour_space_single(include_lab=False))

| Video | Space | same | diff | gap |
|---|---|---|---|---|
| dev_01 | HSV | 0.5900 | 0.3777 | 0.2123 |
| dev_01 | RGB | 0.7529 | 0.6740 | 0.0789 |
| dev_02 | HSV | 0.8724 | 0.5862 | 0.2862 |
| dev_02 | RGB | 0.9323 | 0.7688 | 0.1636 |


HSV's gap is 2.7× wider on dev_01 and 1.75× wider on dev_02 than RGB's.
**But the gap is measured on a single pair, and this is not a conclusive
test**: the final choice is decided at matrix level in Section 5.1.4,
where the F1 and the plateau width are compared across all pairs of
shots. The matrix-level result there is counter-intuitive (RGB actually
achieves a *higher* F1 than HSV) but reveals a fragility that the gap
alone cannot capture.

![Histogram similarity example](./images/t6_hist_example_dev_01.png)

*Figure 2 — Colour similarity of two keyframes from the same scene (`dev_01`, scene `S00`). Top: the two keyframes. Bottom: the marginal distributions of H, S and V.*

**Decision.** We adopt **HSV with `8³`** as the production colour
representation. This is not the strictly optimal choice on any single
metric. `16³` ties `8³` on the mean gap (see above) and Lab has a
larger gap on dev_01 (Section 8.2), but it is the configuration that
**combines a competitive gap on both videos with the lowest
computational cost among the tested variants and it is the most
consistent colour space across the two development videos** (Lab's gap
differs by 0.10 between dev_01 and dev_02; HSV's by only 0.07). The
final matrix-level comparison with Lab is performed in Section 5.1.4,
where F1 and plateau width are computed on the full shot × shot
matrix instead of a single pair.


#### 4.2.2 Local-feature similarity (SIFT)

The colour similarity of 4.2.1 captures the overall distribution of pixels
across HSV cells but is blind to spatial structure. Two images with
inverted layouts may share an almost identical histogram. 

**Pipeline.** For each keyframe pair:

1. Detect keypoints and compute 128-dimensional descriptors on the
   grayscale keyframe (`cv2.SIFT_create(nfeatures=500)`).
2. `BFMatcher(NORM_L2).knnMatch(des1, des2, k=2)`: for each descriptor
   in `kf1`, find the two nearest descriptors in `kf2`.
3. **Lowe's ratio test**: keep a match only if
   `d(NN1) / d(NN2) < 0.75`.
4. The **similarity** is the number of surviving matches, an unbounded
   integer. Aggregation of the K_a × K_b pairwise counts into a shot-to-
   shot value then depends on the aggregator:
   - `max` and `mean` operate on the raw counts and return a value in
     **count units**; `group_shots` is therefore tuned with a threshold
     in those same units (e.g. `avg_T = 42.92` for `sift uniform K=5`
     in Section 5.2).
   - `vote` uses a **fixed internal threshold of 0.5 on the similarity
     values**. That threshold is meaningful only on a `[0, 1]` scale
     (like the colour-histogram output); on raw SIFT counts it would
     never fire, because "half a match" does not occur. `vote` would
     therefore require normalising the SIFT counts to `[0, 1]` first
     (e.g. `s / min(len(kp1), len(kp2))`, as in Section 8.1), a step
     that introduces its own bias (Section 5.1.3).
   - For these reasons SIFT is evaluated only with `max` (Section 5.2),
     keeping the aggregator consistent with the colour-histogram
     pipeline of Section 5.1.2. `mean` would work in principle but is
     not tested with SIFT in this report.

![SIFT matches example](./images/t6_sift_matches_dev_01.png)

*Figure 3: SIFT matches between two keyframes from the same scene
(`dev_01`, scene `S00`) after Lowe's ratio test, with `nfeatures=500`.
Lines connect descriptor correspondences; the number of surviving
matches (15) is the similarity metric used by `sift_similarity`.*

**Parameter choices:** `nfeatures=500` and `ratio=0.75`. Both justified
empirically in **8.4**.

### 4.3 Task 7: Scene grouping

The two similarity measures of Section 4.2 produce a scalar value for
each pair of shots. The final step is to turn
this **N × N similarity matrix** into a **partition of the N shots
into scenes**.

**Pipeline.** The grouping itself is a two-stage process:

1. **`build_similarity_matrix(frames, shots, config)`**: produces the
   N × N float32 matrix `sim[i, j]`, where `sim[i, i] = 1.0` and
   `sim[i, j] == sim[j, i]`. The matrix is populated by extracting
   keyframes per shot (Section 4.1), computing per-keyframe similarities
   with the selected method (Section 4.2) and aggregating the K_a × K_b
   pairwise values into a single shot-to-shot score.

2. **`grouping.group_shots(sim, threshold)`** (given in the starter
   code).

The public entry point `scenes_from_shots(frames, gt_shots, config)`
wraps both stages and returns `{str(shot_id): int_label}`, in the exact
format that `metrics.evaluate_video` expects.

**Complexity consideration.** The pre-detection of SIFT descriptors
inside `build_similarity_matrix` is the single largest optimisation in
the pipeline. The matrix is built **per video**, so the relevant N is
the number of shots in the video (N=37 for dev_01, N=35 for dev_02),
not the total across both videos. Taking the larger of the two (N=37)
with K=5 keyframes per shot, the naive approach would run
`N · N · K · K · 2` SIFT detections:

    37² × 5² × 2 = 68 450

By detecting each keyframe exactly **once** and reusing its descriptors
across every pair it participates in, the total drops to `N · K`
detections:

    37 × 5 = 185

a **×370 reduction**. This keeps the whole run well inside the 5-minute
budget, even when the SIFT-based methods are used. (For dev_02 the
numbers scale to 61 250 naive and 175 with the cache.)

**Threshold selection.** The choice of `threshold` is the single most
consequential decision of Task 7. A low threshold over-merges and a high 
threshold under-merges. Because the optimal threshold depends
on the similarity distribution produced by the selected method, we
**defer the decision to Section 5.1**, where it is tuned empirically
using the sweep procedure described below. **Section 5.2** compares
the 2×2 combination of {hist, sift} × {1, K} keyframes and
**Section 5.1.3** evaluates whether the combined similarity of
Section 8.1 provides any benefit over the isolated signals.

![Similarity matrix vs ground truth](./images/t7_matrix_hist_dev_01.png)

*Figure 4 — Predicted shot × shot similarity matrix (left) and
ground-truth same-scene matrix (right), for `dev_01` under the
production configuration.*


## 5. Results (development and test)

### 5.1 Hyperparameter tuning on the development set

#### 5.1.1 K-sweep

The initial K=3 (Section 4.1) was chosen from coverage/cost bounds
alone. To justify it empirically, we swept K ∈ {1, 3, 5, 8} using
`method="hist"` and additionally compared to the `middle` strategy 
(single centered frame):


In [21]:
show(part3_exp.table_k_sweep())

| Strategy | K | d01_F1 | d02_F1 | mean F1 | width (d01/d02) | time |
|---|---|---|---|---|---|---|
| middle | 1 | 0.889 | 0.923 | 0.906 | 0.08 / 0.00 | 393 ms |
| uniform | 1 | 1.000 | 0.955 | 0.977 | 0.09 / 0.02 | 382 ms |
| uniform | 3 | 1.000 | 0.950 | 0.975 | 0.08 / 0.14 | 4.1 s |
| uniform | 5 | 1.000 | 0.976 | 0.988 | 0.14 / 0.11 | 8.7 s |
| uniform | 8 | 1.000 | 0.976 | 0.988 | 0.15 / 0.12 | 24.5 s |


The `width` column reports the width of the F1 plateau from the threshold
sweep. A wide plateau means the exact choice of T matters less, because a 
range of thresholds achieves the maximum F1.

**The F1 curve is not monotone and the coverage prior is not
confirmed.** Going from K=3 to K=1 actually *raises* mean F1 slightly
(0.975 → 0.977) and K=3 is worse than K=1 on dev_02 specifically
(0.950 vs 0.955). So the "three phases" argument of Section 4.1 is not
what the data supports: **a single keyframe at the start of the shot
is as informative as three evenly-spaced keyframes.** A plausible
explanation is that the first frame is an *establishing frame* (camera
settled, no motion), whereas mid-shot keyframes often contain moving
objects or occlusions.

**Where K=5 actually wins: plateau width.** Applying the robustness
criterion of Section 5.1.4, *F1 within 0.02 of the best, plateau
width ≥ 0.10 on both videos, then prefer the simplest*, to the four
uniform rows of the sweep:

| K | mean F1 | Within 0.02 of best (0.988)? | Min width | Passes? |
|---|---|---|---|---|
| 1 | 0.977 | ✓ (0.011 < 0.02) | **0.02** | ✗ |
| 3 | 0.975 | ✓ (0.013 < 0.02) | **0.08** | ✗ |
| 5 | 0.988 | ✓ (best) | 0.11 | ✓ |
| 8 | 0.988 | ✓ (best) | 0.12 | ✓ |

**K=1 and K=3 are excluded by the width requirement** (0.02 on dev_02
and 0.08 on dev_01, respectively), not by their mean F1. **K=5 and K=8 both pass; 
K=5 is chosen because it is the cheaper of the two** (8.7 s vs 24.5 s, a 2.8× cost 
difference for no measurable gain).

**Decision.** Move from K=3 to **K=5** in the production configuration. 
The cost (8.7 s per video) remains well within the 5-minute budget when 
combined with the fastest method (hist).

#### 5.1.2 Aggregator sweep (max / mean / vote)

For two shots with K_a and K_b keyframes, we obtain K_a × K_b pairwise
similarities. These must be collapsed
into a single shot-to-shot value. Three aggregators are available:

- **`max`** — the strongest pair. A single good keyframe pair is enough
  to link two shots from the same scene.
- **`mean`** — the arithmetic average across all K² pairs.
- **`vote`** — the count of pairs whose similarity exceeds a fixed internal
  threshold (0.5).

We swept the three aggregators on both development videos, keeping
`method="hist"`, `uniform K=5`, and `bins=8³` fixed:

In [22]:
show(part3_exp.table_aggregator_sweep())

| Aggregator | d01 F1 | d02 F1 | mean F1 | width (d01/d02) |
|---|---|---|---|---|
| max | 1.000 | 0.976 | 0.988 | 0.14 / 0.11 |
| mean | 0.952 | 0.952 | 0.952 | 0.00 / 0.03 |
| vote | 0.119 | 0.708 | 0.413 | 3.79 / 2.78 |

**`max` has the highest F1 on both videos.** Mean F1 is 0.988 vs
0.952 (`mean`) and 0.413 (`vote`). Among the aggregators on the same
scale (`max` vs `mean`, both producing similarities in `[0, 1]`),
`max` also has the wider plateau (0.14/0.11 vs 0.00/0.03). The
`vote` plateau widths (3.79/2.78) are in integer-count units (0 to
K²=25) and are therefore **not comparable** to the `[0, 1]` scale.

**Decision.** `max` is retained in the production configuration. Its
correctness is supported by the theoretical argument (a single good
keyframe pair is sufficient to identify a recurring scene) and by
three empirical signals: (i) the **highest mean F1** (0.988 vs 0.952
and 0.413), (ii) the **widest plateau among the aggregators on the
same scale** (0.14/0.11 vs 0.00/0.03 for `mean`), and (iii) no
dependence on a hidden second threshold (unlike `vote`, which has an
internal 0.5 that would also need tuning).

#### 5.1.3 Combined similarity 

Section 8.1 evaluated the blend `α · h + (1−α) · ŝ` on a single test
pair and deferred the choice of α to the full matrix. We now sweep
α ∈ {0.00, 0.25, 0.50, 0.75, 1.00} on the shot × shot matrix, keeping
`uniform K=5`, `aggregator=max`, and `bins=8³` fixed. 
`α=1.00` reproduces the histogram-only configuration of Section 5.1.2 exactly,
which validates the implementation. 

The timing row for `α=1.00` (~11 s) is **not** comparable to the
`method="hist"` rows of Sections 5.1.4 and 5.2 (~8 s): in
`build_similarity_matrix`, SIFT descriptors are pre-detected
**whenever `method="combined"`, regardless of `α`**. With `α=1.0`
the similarity function returns immediately after the histogram, but
the SIFT detection cost has already been paid upstream. The ~11 s
therefore measures SIFT detection (which is unavoidable in the
`combined` path) plus the histogram, not the histogram alone; the true 
histogram-only runtime is the ~9–10 s reported in 5.1.4 and 5.2. 
The α=0.25–0.75 rows cost ~30 s because they pay for both SIFT detection
**and** the K_a × K_b keyframe matching, which the endpoints avoid.


In [23]:
show(part3_exp.table_alpha_sweep())

| α | d01 F1 | d02 F1 | mean F1 | d01 width | d02 width | time |
|---|---|---|---|---|---|---|
| 0.00 | 0.889 | 0.895 | 0.892 | 0.07 | 0.01 | 12.8 s |
| 0.25 | 1.000 | 0.976 | 0.988 | 0.11 | 0.10 | 28.0 s |
| 0.50 | 1.000 | 0.976 | 0.988 | 0.21 | 0.13 | 28.5 s |
| 0.75 | 1.000 | 0.976 | 0.988 | 0.17 | 0.12 | 28.8 s |
| 1.00 | 1.000 | 0.976 | 0.988 | 0.14 | 0.11 | 11.2 s |


**A four-way tie in F1.** Every α ≥ 0.25 achieves the same mean F1 of
0.988. The blend does not improve accuracy over the histogram alone.
The opposite endpoint `α=0.00` corresponds to a **normalised** SIFT-only
configuration (`s / min(len(kp1), len(kp2))`), with the count capped at
the number of features detected in the smaller keyframe and its mean
F1 of 0.892 is 10% below the histogram-only value.

**The raw `sift_similarity` of Section 5.1.6 reaches 0.935** (only 5%
below the histogram). The gap between 0.892 and 0.935 isolates the
cost of normalising SIFT to `[0, 1]`: with `nfeatures=500`, a keyframe
with only 300 detected features caps `s / min(kp)` at 1.0 even for
a perfect match, but a keyframe with 146 features caps it at a much
tighter ceiling. A 15-match keyframe pair yields 0.103 under
normalisation, while the raw count reports the full 15. The `max`
aggregator then compares these normalised values across pairs and
the aggressive rescaling amplifies noise in the well-detected
keyframes. Raw SIFT counts avoid this because the magnitude itself
carries the discriminative signal.

**Decision.** Production uses `method="hist"`, which is equivalent to
`combined` with `α=1.00`. The code's `combined_alpha` default of `0.5`
is only used when `method="combined"` is explicitly selected; under the
production configuration it is never read (see the `combined_alpha` row
in the final-configuration table of Section 5.1.7).

#### 5.1.4 Colour space on the matrix (HSV vs Lab vs RGB)

Section 8.2 compared HSV, Lab, and RGB on a single test pair and found
mixed evidence. That experiment measured the *discrimination gap* on one pair,
not the final grouping performance. To decide with the full shot × shot
matrix, we now repeat the comparison with `uniform K=5`, `aggregator=max`,
and the three colour spaces:


In [24]:
show(part3_exp.table_colour_space_matrix())

| Method | d01 F1 | d02 F1 | mean F1 | d01 width | d02 width | time |
|---|---|---|---|---|---|---|
| hist (HSV) | 1.000 | 0.976 | 0.988 | 0.14 | 0.11 | 9.6 s |
| lab | 1.000 | 0.976 | 0.988 | 0.06 | 0.04 | 11.8 s |
| rgb | 1.000 | 1.000 | 1.000 | 0.06 | 0.00 | 5.6 s |


**RGB appears to win on F1**: a perfect 1.000 on both videos, higher
than HSV's 0.988. **But this F1 is achieved with a plateau of width
0.00 on dev_02 (a single specific threshold)**. The matrix
produced by RGB has same-scene and different-scene distributions so
close that only one exact T separates them; any other threshold
collapses the F1. The 1.000 is therefore an **overfit to the
development set**: the reported F1 would not survive a small shift in
the test videos' similarity distribution. The plateau width is what
catches this failure mode, as the F1 metric alone does not.

**HSV and Lab tie on mean F1 (0.988) but HSV is more robust.**
HSV's plateau is 0.14/0.11, roughly 2-3× wider than Lab's (0.06/0.04).
A wider plateau means the exact T matters less: on unseen test videos,
the true optimal threshold is more likely to fall inside HSV's plateau
than inside Lab's narrower one.

**Decision.** We adopt a two-part criterion that will be used
consistently across the hyperparameter sweeps: **among configurations
whose F1 is within 0.02 of the best, require a plateau width ≥ 0.10 on
both videos to be considered robust. Among those that pass, prefer the
simplest.** The 0.02 tolerance allows slight F1 variations to be
considered "practically tied", treating 1.000 and 0.988 as the same
performance level, while still rejecting any F1 substantially below
the best.

Under this criterion:

| Config | mean F1 | Within 0.02 of best? | Min width | Passes width? |
|---|---|---|---|---|
| HSV | 0.988 | ✓ (0.988 ≥ 1.000 − 0.02) | 0.11 | ✓ |
| Lab | 0.988 | ✓ | 0.04 | ✗ (below 0.10) |
| RGB | 1.000 | ✓ (equal to best) | **0.00** | ✗ (single threshold) |

**Both Lab and RGB fail the width requirement.** Lab by a small margin
(0.04 < 0.10), RGB by the full collapse (0.00).
**HSV is the unique configuration that combines a competitive F1 with
substantial robustness to threshold drift.** HSV is therefore retained.

#### 5.1.5 Bin resolution on the matrix (4³ vs 8³ vs 16³ vs 30·32²)

Section 4.2.1 swept the histogram bin resolution on a single keyframe
pair and found that finer bins reduced the discrimination gap.
That analysis measured the *gap between the same-scene and different-scene means*
on one pair. We now repeat the sweep on the **full shot × shot matrix** to see 
whether the same conclusion holds when the metric is the F1 of the final grouping:


In [25]:
show(part3_exp.table_bins_matrix())

| Bins | d01 F1 | d02 F1 | mean F1 | d01 width | d02 width | mean width | time |
|---|---|---|---|---|---|---|---|
| 4³ | 1.000 | 0.976 | 0.988 | 0.081 | 0.040 | 0.060 | 9.4 s |
| 8³ | 1.000 | 0.976 | 0.988 | 0.141 | 0.111 | 0.126 | 11.5 s |
| 16³ | 1.000 | 0.976 | 0.988 | 0.192 | 0.141 | 0.166 | 11.3 s |
| 30·32² | 1.000 | 0.976 | 0.988 | 0.232 | 0.130 | 0.181 | 10.3 s |

| Bins | d01 width | d02 width | Mean width | vs 8³ |
|---|---|---|---|---|
| 4³ | 0.081 | 0.040 | 0.060 | −52 % |
| 8³ | 0.141 | 0.111 | 0.126 | baseline |
| 16³ | 0.192 | 0.141 | 0.166 | +32 % |
| 30·32² | 0.232 | 0.130 | 0.181 | +44 % |


**A four-way F1 tie.** Every bin resolution
yields identical F1 (1.000 on dev_01, 0.976 on dev_02, mean 0.988). 
This is the most surprising result of the whole hyperparameter sweep, 
because it appears to contradict Section 4.2.1. 

**Section 4.2.1.** On a single pair, finer bins
push both the same-scene and the different-scene similarities
*downward*, so the absolute gap shrinks. On the full
matrix, this downward shift affects *both* classes roughly equally, so
the *ordering* of the pairs is preserved and the F1 stays identical.
What matters for the grouping is not the absolute value of any single
similarity, but the **relative ordering** of the pairs: as long as
same-scene pairs keep scoring above different-scene pairs, the
threshold sweep can find a cut point that separates them and the F1
is preserved.

**A bonus that emerges from the wider-plateau effect.** Although F1 is
unchanged, the **mean plateau width grows with finer bins**. The per-video
widths are **not strictly monotone**. On dev_02 the width dips slightly
from 0.141 (16³) to 0.130 (30·32²), but the mean increases at every
step (the second of the two tables above).

The non-monotonicity on dev_02 is visible at a glance in the second table above:
the mean is monotone because the dev_01 and dev_02 widths move in opposite directions at the
last step (dev_01 +0.040, dev_02 −0.011). A wider plateau means the exact
threshold matters less: on unseen test videos, the true optimal threshold
is more likely to fall inside the plateau.

**Why coarser bins hurt** Despite being the "smoothest"
histogram, 4³ yields the **narrowest plateau** (0.06 mean width).

**Applying the robustness criterion of Section 5.1.4.** Four
configurations tie on F1: `4³`, `8³`, `16³`, `30·32²`. Of these, `4³`
fails the requirement (dev_02 width = 0.040, below 0.10) and is
excluded. The remaining three (`8³`, `16³`, `30·32²`) all pass the
width requirement on both videos. **Among those that pass, we prefer
the simplest**:`8³`, because it is the cheapest to compute (512
bins vs 4 096 and 30 720) and it is the configuration already validated
by every other hyperparameter sweep. The wider plateaus of `16³` and
`30·32²` are documented here as a robustness margin that is *available*
but not necessary: any threshold in `[0.838, 0.909]` achieves the
maximum F1 on both videos already under `8³`.

**Decision.** `bins=(8, 8, 8)` is retained in the production
configuration. It delivers the same F1 as any other tested resolution,
its plateau is already wide enough for practical purposes (both
per-video widths exceed the 0.10 requirement of Section 5.1.4), and it
is the cheapest of the passing configurations.

#### 5.1.6 RANSAC on the matrix — does geometric filtering help?

Section 8.3 explored RANSAC on a single test pair and found a
**regime-dependent effect**: on the dev_01 pair it raised the gap
(+3), on the clean dev_02 pair it degraded it.
The conservative decision from the single-pair experiment was to disable
RANSAC. We now repeat the comparison on the full shot × shot matrix to
see whether the same conclusion holds at distribution level. Since RANSAC
only applies to SIFT-based methods, the sweep uses `method="sift"` and
`method="combined"` (α=0.5), both with `uniform K=5` and `aggregator=max`:


In [26]:
show(part3_exp.table_ransac_matrix())

| Method | RANSAC | d01 F1 | d02 F1 | mean F1 | d01 width | d02 width | time |
|---|---|---|---|---|---|---|---|
| sift | ✗ | 0.947 | 0.923 | 0.935 | 23.70 | 21.39 | 14.3 s |
| sift | ✓ | 0.947 | 0.950 | 0.949 | 31.60 | 0.00 | 22.4 s |
| combined | ✗ | 1.000 | 0.976 | 0.988 | 0.212 | 0.133 | 29.2 s |
| combined | ✓ | 1.000 | 0.976 | 0.988 | 0.192 | 0.097 | 37.2 s |


**Two counter-intuitive findings, both of which confirm the decision to
disable RANSAC in production.**

**Finding 1: RANSAC helps `sift` at F1 but destroys robustness.**
On `method="sift"`, RANSAC improves mean F1 from 0.935 to 0.949 (+1.4%).
This appears to *contradict* the Section 8.3 conclusion, which reported
RANSAC as degrading the dev_02 pair by −19 matches. The key is the
plateau width: on dev_02 the plateau **collapses to width = 0.00**.

**Finding 2: RANSAC is useless for `combined` and slightly harmful.**
On `method="combined"`, RANSAC leaves F1 unchanged (0.988 both ways)
while *narrowing* the plateau from 0.212/0.133 to 0.192/0.097 and *increasing* 
the runtime by ~27% (29.2 s → 37.2 s). RANSAC adds nothing
to the combined similarity. the SIFT term is a small perturbation
and RANSAC only makes it less diverse. 

**Decision.** `use_ransac=False` is retained in the production
configuration. For the production method (`hist`), RANSAC is
irrelevant. For `sift` and `combined`, the plateau width confirms the
single-pair conclusion of Section 8.3. 

#### 5.1.7 Final configuration

Sections 5.1.1 through 5.1.6 swept seven independent design choices.
The consolidated production configuration is:

| Parameter | Value | Justified in |
|---|---|---|
| `method` | `"hist"` (HSV χ²) | 5.1.3 (α=1.00 tie), 5.1.4 (HSV retained) |
| `kf_strategy` | `"uniform"` | 5.1.1 (outperforms `middle`) |
| `K` | `5` | 5.1.1 (smallest K whose plateau passes the width criterion; K=8 adds cost only) |
| `aggregator` | `"max"` | 5.1.2 (mean F1 = 0.988 vs 0.952 and 0.413) |
| `bins` | `(8, 8, 8)` | 5.1.5 (F1 tie, plateau already wide enough) |
| `lowe_ratio` | `0.75` | 8.4 (largest ratio with `same/diff` > 1.3 on the harder video; retains most matches among the equivalent options) |
| `nfeatures` | `500` | 8.4 (`same/diff` 8.25 → 11.5 on dev_02; `unlimited` adds only +6 matches) |
| `use_ransac` | `False` | 5.1.6 (plateau collapse with RANSAC on) |
| `transform` | `"homography"` | irrelevant with `use_ransac=False` |
| `combined_alpha` | `0.5` (code default for `method="combined"`; never read with `method="hist"`) | 5.1.3 (production = `hist`, equivalent to `α=1.00`) |
| `threshold` | `0.87` | plateau analysis of 5.1.1 (see below) |

The four selections above (`K`, `aggregator`, `method`, `bins`) all
follow the same criterion, introduced in Section 5.1.4: **among
configurations that tie on F1, require a plateau width ≥ 0.10 on both
videos to be considered robust; among those that pass the requirement,
prefer the simplest.** This formalises what was already implicit in
the choices above: K=5 was preferred over K=8 (both pass the width 
requirement, but K=5 is ~2.8× faster) and `8³` was preferred over `16³`
and `30·32²` (all three pass, but `8³` is the cheapest). For the
aggregator and colour-space sweeps the winner is unique among those
that pass the width requirement.

**Threshold selection.** Unlike the other parameters, the threshold is
not derived from a dedicated sweep but from the **plateau analysis of
the K-sweep**. For the chosen configuration (hist, uniform K=5, max
aggregation), the optimal-F1 threshold plateaus are:


In [27]:
show(part3_exp.table_threshold_plateau())

| Video | Optimal T | Plateau width | Plateau range |
|---|---|---|---|
| dev_01 | 0.9091 | 0.1414 | [0.838, 0.980] |
| dev_02 | 0.8538 | 0.1106 | [0.798, 0.909] |


The **intersection** of the two plateaus is [0.838, 0.909] — a 0.071-wide
region where **both videos simultaneously achieve their maximum F1**.
Any threshold in this range is equally good on the development set; we chose `T = 0.87`, 
near the geometric centre of the intersection, to maximise the
distance to either boundary in case the optimal threshold on the test
videos drifts slightly. 

**Note on reported timings.** All per-video times in this section are
single-run wall-clock measurements on the development set. The same
configuration can vary between runs, the same hist, uniform K=5 row 
appears as 8.7 s in 5.1.1, 9.6 s in 5.1.4 and 9.8 s in 5.2. These
differences are considered measurement noise, not algorithm differences. Only the
ratios that differ by more than ~50% (e.g. `combined` vs `hist`, or
RANSAC on vs off) are large enough to be interpreted as real cost
differences.

### 5.2 Shot similarity — the 2×2 comparison

Section 4.2 introduced two shot similarity measures (colour histograms
and SIFT local features) and Section 4.1 introduced two keyframe
strategies (`middle` (1 frame) and `uniform` (K frames)). The assignment
requires their pairwise comparison, so we swept all four combinations
on both development videos with the tuned production parameters

`method` and `kf_strategy` according to the row, `uniform K=5`, `K=1 for middle rows`, `aggregator=max`,
`bins=8³`. The threshold is the plateau centre reported per row.
Two additional rows (`combined similarity` of Section 8.1) are shown
as extras.

**Precision, recall and F1** are computed with the given
`metrics.pairwise_scene_counts` on the full shot × shot matrix, after
applying `grouping.group_shots` with the optimal threshold for each
configuration. The `avg_T` column is the mean of the two per-video
plateau centres.


In [28]:
show(part3_exp.table_2x2())

| Method | KF | K | d01 P | d01 R | d01 F1 | d02 P | d02 R | d02 F1 | mean F1 | time |
|---|---|---|---|---|---|---|---|---|---|---|
| hist | `middle` | 1 | 1.000 | 0.800 | 0.889 | 1.000 | 0.857 | 0.923 | 0.906 | 367 ms |
| hist | `uniform` | 5 | 1.000 | 1.000 | 1.000 | 1.000 | 0.952 | 0.976 | 0.988 | 9.8 s |
| sift | `middle` | 1 | 1.000 | 0.700 | 0.824 | 1.000 | 0.762 | 0.865 | 0.844 | 990 ms |
| sift | `uniform` | 5 | 1.000 | 0.900 | 0.947 | 1.000 | 0.857 | 0.923 | 0.935 | 13.4 s |
| combined | `middle` | 1 | 1.000 | 0.800 | 0.889 | 0.952 | 0.952 | 0.952 | 0.921 | 1.7 s |
| combined | `uniform` | 5 | 1.000 | 1.000 | 1.000 | 1.000 | 0.952 | 0.976 | 0.988 | 29.1 s |


**Interpreting the 2×2.** Two conclusions emerge, both consistent with
the dedicated sweeps of Section 5.1:

1. **`uniform K=5` dominates `middle K=1` for every similarity method.**
   The 2×2 column pairs isolate the keyframe strategy effect:
   - `hist`: +0.082 mean F1 (0.906 → 0.988)
   - `sift`: +0.091 mean F1 (0.844 → 0.935)
   - `combined`: +0.067 mean F1 (0.921 → 0.988)

   The gain is consistent across both videos. This is the same conclusion
   already drawn in Section 5.1.1 with a larger K-sweep.

2. **`hist` and `combined` tie on F1 (0.988), but `hist` is ~3× faster**
   (9.8 s vs 29.1 s per video, consistent with the ~2.5× measured 
   independently in Section 5.1.3, within the run-to-run timing noise 
   noted in 5.1.7). The hist method therefore
   dominates the combined similarity at production scale: adding SIFT
   on top of the histogram does not improve accuracy and it multiplies
   the runtime by a factor of three.

**A weaker result for `sift` alone.** SIFT-`uniform` reaches only
0.935 mean F1 vs `hist`-`uniform`'s 0.988. The 5% gap is consistent
with the single-pair observation of Section 4.2.2.

**Zero false positives for the `uniform K=5` configurations.** All
three `uniform K=5` rows (hist, sift, combined) achieve P=1.000 on
both videos: with the extra keyframes, no pair of different scenes is
ever merged and their errors are exclusively missed pairs (FN), not
spurious merges (FP). The only configuration with a non-perfect
precision is `combined middle` on dev_02 (P=0.952, one spurious
merge), where a single mid-shot keyframe occasionally produces a
false match. The `middle` rows for hist and sift still reach P=1.000,
so the spurious merge is a weakness specific to combining a noisy
signal (normalised SIFT) with a single non-establishing keyframe, not
a general failure of the pipeline. 

**Storyboard evidence.** The two storyboards below show the predictions
of the winning configuration on the two development videos. Each tile
is the middle keyframe of a shot, labelled with the scene id predicted
by `group_shots`. **Same-colour tiles that depict the same physical
scene are correct groupings; tiles that share a colour but depict
different content (or vice versa) are errors.**

![Storyboard dev_01](./images/t7_storyboard_hist_uniform_T0.87_dev_01.png)

*Figure 5: Storyboard of `dev_01` with the production configuration
(`hist`, `uniform K=5`, threshold 0.87). The system detects **28 scenes
in 37 shots**, an exact match with the 28 unique scenes in the ground
truth (F1 = 1.000).*

![Storyboard dev_02](./images/t7_storyboard_hist_uniform_T0.87_dev_02.png)

*Figure 6: Storyboard of `dev_02` with the production configuration.
The system detects **20 scenes in 35 shots** against 19 unique scenes
in the ground truth (F1 = 0.976).*

### 5.3 Shot boundary detection on dev 


### 5.4 Scene grouping on dev 


### 5.5 Final results on test_01 + test_02


## 6. Analysis and failure cases

## 7. Conclusion and self-evaluation

## 8. Extras

### 8.1 Combined similarity (colour + features)

**Motivation.** The assignment's extras list
proposes combining them into a single shot similarity.

**Method.** We blend the two normalised similarities with a convex weight
`α ∈ [0, 1]`:

$$
\mathrm{sim}(k_1, k_2) = \alpha \cdot h(k_1, k_2) + (1-\alpha) \cdot \hat{s}(k_1, k_2)
$$

where `h` is `histogram_similarity`, `s` is `sift_similarity`, and
`ŝ = s / min(len(kp1), len(kp2))` normalises the SIFT count to `[0, 1]`.

**Endpoint validation.** `α=1` reproduces `histogram_similarity` exactly,
which validates the implementation. The opposite endpoint `α=0` produces the **normalised**
SIFT count (`s / min(len(kp1), len(kp2))`) and **not** the raw match
count returned by `sift_similarity`. The two are different by
construction: `histogram_similarity` is bounded to `[0, 1]`, while
`sift_similarity` is unbounded, so a convex blend between them requires
normalising SIFT first. `α=0` is therefore a *normalised* SIFT-only
configuration, not `sift_similarity`. The matrix-level comparison of
the two is in Section 5.1.3 (α=0, F1 = 0.892) and Section 5.1.6
(raw SIFT, F1 = 0.935).

**Results** (dev_01 + dev_02 in [0, 1] units):


In [29]:
show(part3_exp.table_alpha_single())

| Video | α=0.00 | α=0.25 | α=0.50 | α=0.75 | α=1.00 |
|---|---|---|---|---|---|
| dev_01 gap | 0.0627 | 0.1001 | 0.1375 | 0.1749 | 0.2123 |
| dev_02 gap | 0.1253 | 0.1656 | 0.2058 | 0.2460 | 0.2862 |


**Interpretation.** The gap grows **linearly** with α. On this specific test pair, 
the histogram dominates.

**But the test pair is not representative.** It compares the *first* and
*last* shot of the same scene. 
The blend's hypothesis (that SIFT and colour fail on disjoint regimes) 
can only be tested on the full shot × shot matrix. 
We therefore defer the α choice to the α sweep on the matrix in Section 5.1.3.
Production uses `method="hist"`, which is equivalent to `combined` with
`α=1.00` (Section 5.1.3); the code's `combined_alpha=0.5` is the default
only when `method="combined"` is selected explicitly and is never read
under the production configuration.


### 8.2 Colour space comparison (Lab vs HSV vs RGB)

**Motivation.** 4.2.1 justifies HSV through theoretical considerations
(chromaticity decoupled from luminance). But does Lab
discriminate better than HSV on real footage?

**Method.** The same χ² and 3D-histogram pipeline, applied in three
colour spaces. The colour space is a parameter of the shared helper
`_colour_histogram(bgr, space, bins)` and three public wrappers
(`histogram_similarity` for HSV `lab_similarity` for Lab and
`rgb_similarity` for RGB) call it with the corresponding `space`. In
the pipeline, the choice is made by the `method` field of the config
(`"hist"`, `"lab"` or `"rgb"`), which `_pairwise_keyframe_similarities`
maps to the corresponding wrapper.

**Results:**


In [30]:
show(part3_exp.table_colour_space_single(include_lab=True))

| Video | Space | same | diff | gap |
|---|---|---|---|---|
| dev_01 | HSV | 0.5900 | 0.3777 | 0.2123 |
| dev_01 | Lab | 0.7836 | 0.5017 | 0.2819 |
| dev_01 | RGB | 0.7529 | 0.6740 | 0.0789 |
| dev_02 | HSV | 0.8724 | 0.5862 | 0.2862 |
| dev_02 | Lab | 0.9213 | 0.7427 | 0.1786 |
| dev_02 | RGB | 0.9323 | 0.7688 | 0.1636 |
| mean | HSV | — | — | 0.2493 |
| mean | Lab | — | — | 0.2302 |
| mean | RGB | — | — | 0.1212 |


**Interpretation.** The result is **mixed and symmetric**: Lab has the
larger gap on dev_01 (+33% over HSV), HSV has the larger gap on dev_02
(+60% over Lab). The two signals are of the same kind (gap between
same-scene and different-scene on a single keyframe pair) and each
colour space wins on exactly one video. **Neither is uniformly better
on this single pair.**

**What this experiment cannot decide.** A gap measured on a single
keyframe pair is too noisy to establish a ranking: the mean gap is 
HSV 0.2493 vs Lab 0.2302 (a difference of 0.019), well inside the 
variance one would expect from swapping the keyframe pair. The single
-pair gap is therefore reported as **evidence that both HSV and Lab
are plausible**, not as a decision criterion.

**Decision deferred to Section 5.1.4.** The final choice is made at
matrix level (F1 + plateau width on the full shot × shot matrix), where
HSV and Lab tie on mean F1 (0.988 both) but HSV has a 2–3× wider
plateau (0.14/0.11 vs 0.06/0.04). `lab_similarity` remains in the code
as a competitive alternative; Section 5.1.4 documents why HSV is
retained.


### 8.3 RANSAC transformation model 

**Scope.** The assignment suggests RANSAC + homography for **Part 2**, 
so we tested it in the Task 6 pipeline as an exploration of whether 
geometric filtering could further clean up SIFT matches.

**Method.** Four models, all fitted with RANSAC where applicable, on the
same three keyframes used in 4.2.2 (`nfeatures=500`, `ratio=0.75`):

| transform | DOF | 
|---|---|
| `none` (baseline) | — |
| `translation` | 2 | 
| `similarity` | 4 |
| `affine` | 6 | 
| `homography` | 8 |

**Results:**


In [31]:
show(part3_exp.table_ransac_single())

| Video | transform | same | diff | gap | time |
|---|---|---|---|---|---|
| dev_01 | none | 15 | 11 | 4 | 1.4 ms |
| dev_01 | translation | 8 | 1 | 7 | 2.0 ms |
| dev_01 | similarity | 10 | 3 | 7 | 1.9 ms |
| dev_01 | affine | 11 | 4 | 7 | 2.0 ms |
| dev_01 | homography | 12 | 6 | 6 | 4.6 ms |
| dev_02 | none | 69 | 6 | 63 | 4.1 ms |
| dev_02 | translation | 1 | 0 | 1 | 5.5 ms |
| dev_02 | similarity | 49 | 3 | 46 | 5.0 ms |
| dev_02 | affine | 49 | 4 | 45 | 5.1 ms |
| dev_02 | homography | 49 | 5 | 44 | 6.1 ms |


**Interpretation.**:

1. **dev_01** Baseline has `same=15, diff=11` — barely
   discriminative. RANSAC raises the gap from 4 to 7. The **benefit is 
   real but small** in absolute terms.
2. **dev_02** Baseline has `same=69, diff=6`. RANSAC rejects 20 **true** 
   matches (`same`: 69 → 49) to remove only 1-3 false ones (`diff`: 6 → 3-5).
   The **cost is large**.

The three higher-DOF models (similarity, affine, homography) produce
**nearly identical** results. The extra degrees of freedom
do not add discriminative power here. `translation` fails catastrophically
on dev_02.

**Conclusion.** The conservative default is therefore to
**disable RANSAC for Task 6**. The RANSAC sweep on the matrix in Section 5.1.6 will
reveal the dataset's actual distribution and the decision can be revisited.


### 8.4 SIFT parameter sensitivity (nfeatures and Lowe's ratio)

**Motivation.** The assignment specifies SIFT + Lowe's ratio test for
Task 6 but does not fix the number of keypoints retained (`nfeatures`)
nor the ratio threshold. We swept both on the development videos to
justify the values used in 4.2.2.

**nfeatures**: maximum keypoints retained by SIFT. Sweep over
`{100, 200, 500, unlimited}`:


In [32]:
show(part3_exp.table_nfeatures_single())

| Video | nfeatures | kp_a | kp_b | same | diff | gap | time |
|---|---|---|---|---|---|---|---|
| dev_01 | 100 | 100 | 100 | 7 | 5 | 2 | 61.7 ms |
| dev_01 | 200 | 200 | 146 | 11 | 9 | 2 | 45.1 ms |
| dev_01 | 500 | 307 | 146 | 15 | 11 | 4 | 45.4 ms |
| dev_01 | unlimited | 307 | 146 | 15 | 11 | 4 | 47.8 ms |
| dev_02 | 100 | 100 | 100 | 19 | 0 | 19 | 42.1 ms |
| dev_02 | 200 | 200 | 200 | 33 | 4 | 29 | 44.4 ms |
| dev_02 | 500 | 500 | 500 | 69 | 6 | 63 | 59.8 ms |
| dev_02 | unlimited | 545 | 678 | 75 | 6 | 69 | 71.4 ms |


`nfeatures=500` **more than doubles the discrimination gap on dev_02** relative
to the 200-cap (29 → 63) and the same/diff ratio jumps from 8.25
(33/4) to 11.5 (69/6). Going from 500 to `unlimited` adds only
+6 same-scene matches with no change in the different-scene count
(gap 63 → 69; same/diff 11.5 → 12.5) at the same cost. Following the
`same/diff` criterion introduced below, the +6 matches are within the
natural variance of SIFT detection between frames and do not justify
removing the cap. On dev_01 the two configurations are identical
(only 307 and 146 keypoints are detected, both below 500). We therefore
keep **`nfeatures=500`** as an explicit, reproducible cap, as the
`unlimited` alternative gives no reliable gain over it.

**Lowe's ratio**: the ratio threshold. A sweep over `{0.50, 0.60,
0.70, 0.75, 0.80, 0.90}` with `nfeatures=500` gives the following
same-scene / different-scene match counts on the pair from Section
4.2.2:


In [33]:
show(part3_exp.table_lowe_single())

| ratio | dev_01 (same / diff / gap) | dev_02 (same / diff / gap) |
|---|---|---|
| 0.50 | 8 / 1 / 7 | 32 / 0 / 32 |
| 0.60 | 8 / 2 / 6 | 45 / 1 / 44 |
| 0.70 | 10 / 7 / 3 | 59 / 3 / 56 |
| 0.75 | 15 / 11 / 4 | 69 / 6 / 63 |
| 0.80 | 22 / 21 / 1 | 81 / 11 / 70 |
| 0.90 | 74 / 68 / 6 | 176 / 68 / 108 |

| ratio | dev_01 (same/diff) | dev_02 (same/diff) |
|---|---|---|
| 0.50 | 8.00 | ∞ (0 false) |
| 0.60 | 4.00 | 45.00 |
| 0.70 | 1.43 | 19.67 |
| 0.75 | 1.36 | 11.50 |
| 0.80 | 1.05 | 7.36 |
| 0.90 | 1.09 | 2.59 |


**The dev_01 / dev_02 split explains the shape of the table.** The
`S00` keyframe on dev_01 has only 307 SIFT features (it does not
reach the `nfeatures=500` cap) and its two takes differ substantially
in viewpoint. As the ratio threshold grows, the number of same-scene
matches increases but the number of different-scene matches grows at
a similar pace, so the *separation* between the two classes collapses
above ratio 0.80. On dev_02, the same-scene count grows monotonically with the ratio
and the different-scene count stays at or below 11 up to ratio 0.80; only at
0.90 does it jump to 68, at which point the same/diff ratio collapses
from 7.36 (at 0.80) to 2.59.

**The `same/diff` ratio is the discriminating quantity, not the gap.**
The gap (`same − diff`) scales with the total number of matches: a
ratio threshold that lets 74 vs 68 matches through will have gap = 6,
but the classifier cannot separate same from different scenes when
`same ≈ diff`. The relevant ratios are:

**Three regimes emerge:**

- **0.50 and 0.60** give the cleanest separation (dev_01 ratio 8.00
  and 4.00) but retain only **8 matches** in the harder keyframe.
  With this few features, matrix-level thresholding is fragile:
  keyframes of different shots have very different feature counts
  (307 vs 146 on the `S00` pair), and an 8-match keyframe cannot be
  tuned reliably.

- **0.70 and 0.75 are equivalent in gap on both videos** (dev_01:
  3 vs 4; dev_02: 56 vs 63), with a solid `same/diff` ratio on
  dev_01 (1.43 vs 1.36). We keep **0.75** as the conventional
  value (Lowe's original recommendation), as it **retains more
  matches** (15 vs 10 on dev_01; 69 vs 59 on dev_02), which is
  preferable when the same keyframe is matched against many others
  at matrix level.

- **0.80 and 0.90 are rejected.** At 0.80 the dev_01 gap shrinks to
  1 and the `same/diff` ratio falls to **1.05**, the two classes
  overlap almost completely (22 same vs 21 diff). At 0.90 the
  *absolute* gap on dev_01 grows back to 6 (74 vs 68), but the
  `same/diff` ratio is only **1.09**: 68 different-scene matches
  slip through for every 74 same-scene matches. 

**Decision.** **We keep `ratio=0.75`.** The largest ratio with a
`same/diff` separation above 1.3 on the harder development video
and among the equivalent alternatives it is the one that retains
the most matches for matrix-level thresholding.

## 9. Use of AI tools